# Wheat Nutrient Mapping — GBR Baseline + ResUNet (Kaggle GPU)

**Этап 1 — GBR Baseline:** попиксельное предсказание через GradientBoosting на 17 признаках (5 каналов + 12 индексов). Без GPU.
**Этап 2 — ResUNet:** обучение регрессионной U-Net с предобученным ResNet34 на патчах 128×128.

**Вход (17 каналов):** 5 каналов + 12 индексов из корреляционного анализа  
**Веса loss:** из R² улучшенного ML-анализа (07b_ml_enhanced_multi)  


In [ ]:
!pip install segmentation-models-pytorch rasterio geopandas -q

In [ ]:
import os, sys, warnings, time
from pathlib import Path
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import torch
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings('ignore')

DATASET_PATH = Path('/kaggle/input/wheat-spectral-data')  # ← имя вашего датасета
WORK_PATH    = Path('/kaggle/working')
sys.path.insert(0, str(DATASET_PATH))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Verify dataset structure
print(f'\nDataset path exists: {DATASET_PATH.exists()}')
if DATASET_PATH.exists():
    print(f'Contents: {[p.name for p in DATASET_PATH.iterdir()][:10]}')


In [ ]:
from analysis.cfg import load_config, short_name
from analysis.loaders import load_chemistry
from analysis.cv_pipeline import (
    LOSS_WEIGHTS, SELECTED_INDICES, N_CHANNELS,
    compute_band_stats, extract_patches,
    LabelScaler,
    load_point_coords, world_to_pixel,
    NutrientPatchDataset, build_model,
    weighted_huber_loss, train_epoch, eval_epoch, eval_r2_per_target,
    spatial_cv_splits, compute_metrics,
    predict_full_map_tiled, save_nutrient_geotiff,
)

cfg = load_config(DATASET_PATH / 'config.yaml')

ORTHO_PATH  = str(DATASET_PATH / 'data/ortho/20230619_F14_Micasense.tif')
DATE_KEY    = 'date2'
PATCH_SIZE  = 128
BATCH_SIZE  = 16
EPOCHS      = 200
LR          = 5e-5        # low LR: 17-ch adapted ResNet34 needs careful warmup
PATIENCE    = 25
N_FOLDS     = 5
MIXUP_ALPHA = 0.4
FP16        = False       # FP32: safer with NaN labels and adapted 17-ch input
TILE_SIZE   = 512

print(f'Targets: {cfg["chemistry"]["target_elements"]}')
print(f'Input channels: {N_CHANNELS} (5 bands + {len(SELECTED_INDICES)} indices)')
print(f'LR={LR}  FP16={FP16}  EPOCHS={EPOCHS}')


In [ ]:
# ── Band statistics (strip sampling, no full raster load) ─────────────────
print('Computing band stats...')
band_stats = compute_band_stats(ORTHO_PATH)
print(f'  per-band (p2, p98): {[(round(lo,1), round(hi,1)) for lo,hi in band_stats]}')

with rasterio.open(ORTHO_PATH) as src:
    meta = src.meta.copy()
print(f'  raster: {meta["height"]}×{meta["width"]}  crs={meta["crs"]}')

In [ ]:
# ── Chemistry + point coordinates ─────────────────────────────────────────
dsmap     = cfg['chemistry']['date_sampling_map']
sk        = dsmap.get(DATE_KEY, 'sampling2')
chem      = load_chemistry(cfg['paths']['chemistry'][sk],
                           cfg['chemistry']['columns'],
                           cfg['chemistry']['id_offsets'].get(sk, 0))
avail     = [e for e in cfg['chemistry']['target_elements'] if e in chem.columns]
n_targets = len(avail)
print(f'Targets ({n_targets}): {avail}')

# Find gpkg file for point coordinates
# Try config pattern first, then fallback to recursive search
pattern     = cfg['paths']['multi'].get(DATE_KEY, '')
search_path = str(DATASET_PATH / pattern.lstrip('/').lstrip('\\'))
gpkg_files  = sorted(glob(search_path))
print(f'Pattern: {search_path}')
print(f'Found: {len(gpkg_files)} gpkg files')

if not gpkg_files:
    # Fallback: find any gpkg in dataset
    gpkg_files = sorted(glob(str(DATASET_PATH / '**' / '*.gpkg'), recursive=True))
    print(f'Fallback search found: {len(gpkg_files)} gpkg files')
    if gpkg_files:
        print(f'Using: {gpkg_files[0]}')

if not gpkg_files:
    raise FileNotFoundError(
        f'No .gpkg files found. Check DATASET_PATH and config paths.\n'
        f'DATASET_PATH contents: {list(DATASET_PATH.iterdir())}'
    )

ids, coords   = load_point_coords(gpkg_files[0], target_crs=meta['crs'])
rows_px, cpx  = world_to_pixel(coords, meta['transform'])
H, W          = meta['height'], meta['width']
ok            = (rows_px >= 0) & (rows_px < H) & (cpx >= 0) & (cpx < W)
ids, coords   = ids[ok], coords[ok]
rows_px, cpx  = rows_px[ok], cpx[ok]

common  = np.intersect1d(ids, chem.index.values)
msk     = np.isin(ids, common)
rows_px, cpx  = rows_px[msk], cpx[msk]
chem_sub      = chem.loc[ids[msk]]
coords_sub    = coords[msk]
Y             = chem_sub[avail].values.astype(np.float32)
print(f'Labeled points: {len(common)}')

In [ ]:
# ── Extract patches via windowed reads (no full raster in memory) ──────────
print(f'Extracting {len(common)} patches {PATCH_SIZE}×{PATCH_SIZE}...')
t0 = time.time()
all_patches = extract_patches(ORTHO_PATH, rows_px, cpx, PATCH_SIZE, band_stats)
print(f'  shape: {all_patches.shape}  size: {all_patches.nbytes/1e6:.0f} MB  '
      f'time: {time.time()-t0:.1f}s')

In [ ]:
# ── Diagnostics: check patches and labels for NaN ─────────────────────────
nan_in_patches = np.isnan(all_patches).sum()
inf_in_patches = np.isinf(all_patches).sum()
print(f'Patches — NaN: {nan_in_patches}  Inf: {inf_in_patches}')
print(f'Patches range: [{all_patches.min():.3f}, {all_patches.max():.3f}]')

print('\nLabel stats per nutrient (raw Y):')
for j, e in enumerate(avail):
    col = Y[:, j]
    fin = col[np.isfinite(col)]
    print(f'  {short_name(cfg,e):<6}: n={len(fin):>3}  '
          f'range=[{fin.min():.2f}, {fin.max():.2f}]  '
          f'NaN={np.isnan(col).sum()}')

# Clean patches: replace any residual NaN/Inf
all_patches = np.nan_to_num(all_patches, nan=0.0, posinf=1.0, neginf=-1.0)
print(f'\nAfter clean — NaN: {np.isnan(all_patches).sum()}  '
      f'range: [{all_patches.min():.3f}, {all_patches.max():.3f}]')


In [ ]:
# ── GBR Baseline: train + tiled map prediction ─────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from analysis.cv_pipeline import apply_gbr_tiled, save_nutrient_geotiff, compute_metrics

def train_gbr_models(X, Y, target_names):
    models, scalers = [], []
    for j, name in enumerate(target_names):
        y = Y[:, j]; mask = np.isfinite(y)
        if mask.sum() < 10:
            models.append(None); scalers.append(None); continue
        sc = StandardScaler()
        m  = GradientBoostingRegressor(
            n_estimators=300, max_depth=5,
            learning_rate=0.05, subsample=0.8, random_state=42)
        m.fit(sc.fit_transform(X[mask]), y[mask])
        models.append(m); scalers.append(sc)
        print(f'  {name}: n={mask.sum()}')
    return models, scalers

def extract_point_features_local(all_patches):
    """Extract centre pixel from each patch → (N, 17) feature matrix."""
    cx = all_patches.shape[2] // 2
    cy = all_patches.shape[3] // 2
    return all_patches[:, :, cx, cy]   # (N, 17)

# ── Point-level spatial CV ─────────────────────────────────────────────────
X_pts = extract_point_features_local(all_patches)
print(f'Feature matrix: {X_pts.shape}')

all_true = {n: [] for n in avail}
all_pred = {n: [] for n in avail}

print(f'\nGBR Spatial {N_FOLDS}-fold CV:')
for fold, (tr_idx, val_idx) in enumerate(splits):
    mdls, scs = train_gbr_models(X_pts[tr_idx], Y[tr_idx], avail)
    for j, name in enumerate(avail):
        if mdls[j] is None: continue
        yp = mdls[j].predict(scs[j].transform(X_pts[val_idx]))
        all_true[name].extend(Y[val_idx, j].tolist())
        all_pred[name].extend(yp.tolist())
    r2s = [f"{compute_metrics(np.array(all_true[n]), np.array(all_pred[n]))['R2']:+.3f}" for n in avail[:4]]
    print(f'  Fold {fold+1}: [{', '.join(r2s)}...]')

# ── CV metrics ────────────────────────────────────────────────────────────
print(f'\n{"Nutrient":<20} {"R2":>6} {"RMSE":>8} {"RPD":>6}')
print('-'*44)
baseline_summary = []
for name in avail:
    m  = compute_metrics(np.array(all_true[name]), np.array(all_pred[name]))
    sn = short_name(cfg, name)
    print(f'{sn:<20} {m["R2"]:>+6.3f} {m["RMSE"]:>8.4f} {m["RPD"]:>6.2f}')
    baseline_summary.append({'element': name, **m})

baseline_df = pd.DataFrame(baseline_summary)
baseline_df.to_csv(WORK_PATH / 'baseline_cv_metrics.csv', index=False, float_format='%.4f')
print(f'Saved: baseline_cv_metrics.csv')

# ── Full map prediction (tiled, writes directly to file) ─────────────────
print('\nTraining final GBR models on all data...')
mdls_f, scs_f = train_gbr_models(X_pts, Y, avail)

print('Running tiled map prediction...')
apply_gbr_tiled(
    ORTHO_PATH, mdls_f, scs_f, avail, band_stats, meta,
    out_path=str(WORK_PATH / 'baseline_nutrients.tif'),
    tile_size=512,
)

# ── Scatter plots ─────────────────────────────────────────────────────────
cmaps_s = ['RdYlGn','YlOrRd','Blues','Greens','PuRd',
           'Oranges','BuGn','RdPu','YlGn','BuPu','GnBu','OrRd']
ncols = 4; nrows = int(np.ceil(len(avail)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows))
axes = axes.flatten()
for i, name in enumerate(avail):
    ax = axes[i]
    yt = np.array(all_true[name]); yp_arr = np.array(all_pred[name])
    msk = np.isfinite(yt) & np.isfinite(yp_arr)
    m   = compute_metrics(yt[msk], yp_arr[msk])
    ax.scatter(yt[msk], yp_arr[msk], alpha=0.6, s=25, c='#2E75B6', edgecolors='none')
    lims = [min(yt[msk].min(), yp_arr[msk].min()), max(yt[msk].max(), yp_arr[msk].max())]
    ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)
    ax.set_title(short_name(cfg, name), fontsize=9)
    ax.text(0.05, 0.95, f"R²={m['R2']:.3f}\nRMSE={m['RMSE']:.3f}",
            transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', fc='wheat', alpha=0.5))
    ax.set_xlabel('Observed', fontsize=8); ax.set_ylabel('Predicted', fontsize=8)
    ax.tick_params(labelsize=7)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('GBR Baseline — Spatial CV', fontsize=12)
plt.tight_layout()
fig.savefig(WORK_PATH / 'baseline_scatter_cv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Baseline complete.')


In [ ]:
# ── Normalise labels ──────────────────────────────────────────────────────
label_scaler = LabelScaler().fit(Y)
Y_scaled     = label_scaler.transform(Y)

nan_in_labels = np.isnan(Y_scaled).sum()
print(f'Y_scaled NaN count: {nan_in_labels} (expected — these are masked in loss)')
print('Label ranges after normalisation:')
for j, e in enumerate(avail):
    col = Y_scaled[:, j]; fin = col[np.isfinite(col)]
    if len(fin) > 0:
        print(f'  {short_name(cfg,e):<6}: [{fin.min():.2f}, {fin.max():.2f}]  '
              f'NaN={np.isnan(col).sum()}')


In [ ]:
# ── Loss weights from R² of enhanced ML ───────────────────────────────────
w_arr   = np.array([LOSS_WEIGHTS.get(e, 0.5) for e in avail], dtype=np.float32)
w_arr   = w_arr / w_arr.sum() * n_targets
weights = torch.tensor(w_arr).to(device)

for e, w in zip(avail, w_arr):
    print(f'  {e:<20}: {w:.4f}')

In [ ]:
# ── Spatial CV splits ──────────────────────────────────────────────────────
splits = spatial_cv_splits(coords_sub, n_folds=N_FOLDS)
for f, (tr, val) in enumerate(splits):
    print(f'Fold {f+1}: train={len(tr)}  val={len(val)}')

In [ ]:
# ── Training helper ────────────────────────────────────────────────────────
def run_fold(fold_idx, tr_idx, val_idx):
    print(f'\n[Fold {fold_idx+1}/{N_FOLDS}]  train={len(tr_idx)}  val={len(val_idx)}')

    tr_ds  = NutrientPatchDataset(all_patches[tr_idx], Y_scaled[tr_idx],
                                   augment=True, mixup_alpha=MIXUP_ALPHA)
    val_ds = NutrientPatchDataset(all_patches[val_idx], Y_scaled[val_idx], augment=False)
    print(f'  train patches: {len(tr_ds)}  val patches: {len(val_ds)}')

    tr_loader  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2)

    model = build_model(n_targets, pretrained=True).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch   = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                opt, T_0=max(10, EPOCHS//5), T_mult=2)
    scaler = GradScaler() if FP16 else None

    # Save initial state as fallback before training starts
    best_val    = np.inf
    pat         = 0
    best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    history     = {'train': [], 'val': []}

    for ep in range(1, EPOCHS + 1):
        model.train(); tloss = 0.0
        for patches, labels in tr_loader:
            patches, labels = patches.to(device), labels.to(device)
            opt.zero_grad()
            if FP16:
                with autocast():
                    out  = model(patches)
                    cp   = out[:, :, out.shape[2]//2, out.shape[3]//2]
                    loss = weighted_huber_loss(cp.float(), labels, weights)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                out  = model(patches)
                cp   = out[:, :, out.shape[2]//2, out.shape[3]//2]
                loss = weighted_huber_loss(cp, labels, weights)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            tloss += loss.item()
        tloss /= max(len(tr_loader), 1)

        vloss = eval_epoch(model, val_loader, weights, device)
        sch.step()
        history['train'].append(tloss)
        history['val'].append(vloss)

        if vloss < best_val:
            best_val = vloss; pat = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            pat += 1
            if pat >= PATIENCE:
                print(f'  early stop @ ep {ep}'); break

        if ep % 20 == 0 or ep == 1:
            print(f'  ep {ep:>3}  train={tloss:.4f}  val={vloss:.4f}  '
                  f'best={best_val:.4f}  lr={sch.get_last_lr()[0]:.2e}')

    if best_state is not None:
        model.load_state_dict(best_state)
    r2s = eval_r2_per_target(model, val_loader, device, n_targets)
    print('  Val R²:')
    for i, e in enumerate(avail):
        print(f'    {short_name(cfg, e):<6}: {r2s[i]:+.3f}')

    torch.save(best_state, WORK_PATH / f'ckpt_fold{fold_idx+1}.pt')

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history['train'], label='train', alpha=0.8)
    ax.plot(history['val'],   label='val',   alpha=0.8)
    ax.set_xlabel('epoch'); ax.set_ylabel('Huber loss')
    ax.set_title(f'Fold {fold_idx+1}'); ax.legend(); plt.tight_layout()
    fig.savefig(WORK_PATH / f'loss_fold{fold_idx+1}.png', dpi=120)
    plt.show()

    return r2s

In [ ]:
# ── Run CV ─────────────────────────────────────────────────────────────────
all_r2 = []
for fold, (tr_idx, val_idx) in enumerate(splits):
    all_r2.append(run_fold(fold, tr_idx, val_idx))

all_r2 = np.array(all_r2)   # (n_folds, n_targets)

print('\n' + '='*50)
print('CV R² (mean ± std):')
cv_summary = []
for i, e in enumerate(avail):
    m, s = np.nanmean(all_r2[:, i]), np.nanstd(all_r2[:, i])
    print(f'  {short_name(cfg,e):<6}: {m:+.3f} ± {s:.3f}')
    cv_summary.append({'element': e, 'R2_mean': m, 'R2_std': s})

pd.DataFrame(cv_summary).to_csv(WORK_PATH / 'cv_r2_summary.csv',
                                 index=False, float_format='%.4f')
print(f'Saved: cv_r2_summary.csv')

In [ ]:
# ── Final model on all data ────────────────────────────────────────────────
print('Training final model on all data...')

full_ds     = NutrientPatchDataset(all_patches, Y_scaled, augment=True, mixup_alpha=MIXUP_ALPHA)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True)
print(f'Dataset: {len(full_ds)} patches')

final_model = build_model(n_targets, pretrained=True).to(device)
opt_f       = torch.optim.AdamW(final_model.parameters(), lr=LR, weight_decay=1e-4)
sch_f       = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                  opt_f, T_0=max(10, EPOCHS//5), T_mult=2)
scaler_f    = GradScaler() if FP16 else None

for ep in range(1, EPOCHS + 1):
    final_model.train(); tloss = 0.0
    for patches, labels in full_loader:
        patches, labels = patches.to(device), labels.to(device)
        opt_f.zero_grad()
        if FP16:
            with autocast():
                out  = final_model(patches)
                cp   = out[:, :, out.shape[2]//2, out.shape[3]//2]
                loss = weighted_huber_loss(cp.float(), labels, weights)
            scaler_f.scale(loss).backward()
            scaler_f.unscale_(opt_f)
            torch.nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
            scaler_f.step(opt_f); scaler_f.update()
        else:
            out  = final_model(patches)
            cp   = out[:, :, out.shape[2]//2, out.shape[3]//2]
            loss = weighted_huber_loss(cp, labels, weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
            opt_f.step()
        tloss += loss.item()
    tloss /= max(len(full_loader), 1)
    sch_f.step()
    if ep % 20 == 0 or ep == 1:
        print(f'  ep {ep:>3}  loss={tloss:.4f}')

torch.save(final_model.state_dict(), WORK_PATH / 'model_final.pt')
print('Saved: model_final.pt')

In [ ]:
# ── Tiled map inference (no full raster in memory) ─────────────────────────
print('Predicting full nutrient map (tiled)...')
t0 = time.time()

pred_map = predict_full_map_tiled(
    final_model, ORTHO_PATH, band_stats, meta, device,
    tile=TILE_SIZE, overlap=TILE_SIZE//4,
    batch_size=BATCH_SIZE, fp16=FP16,
)
print(f'Inference done in {time.time()-t0:.1f}s  |  shape: {pred_map.shape}')

# Denormalise predictions back to original units
pred_map = label_scaler.inverse_transform_map(pred_map)
print('Prediction ranges (original units):')
for j, e in enumerate(avail):
    band = pred_map[j]; v = band[np.isfinite(band) & (band != 0)]
    if len(v): print(f'  {short_name(cfg,e):<6}: [{v.min():.3f}, {v.max():.3f}]')

save_nutrient_geotiff(
    pred_map, meta,
    str(WORK_PATH / 'nutrients_date2.tif'),
    avail,
)

In [ ]:
# ── Preview maps ───────────────────────────────────────────────────────────
cmaps = ['RdYlGn','YlOrRd','Blues','Greens','PuRd',
         'Oranges','BuGn','RdPu','YlGn','BuPu','GnBu','OrRd']

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
for i, (ax, name) in enumerate(zip(axes.flatten(), avail)):
    band  = pred_map[i]
    valid = band[np.isfinite(band) & (band != 0)]
    if len(valid) == 0: ax.set_visible(False); continue
    vmin, vmax = np.percentile(valid, 2), np.percentile(valid, 98)
    im = ax.imshow(band, cmap=cmaps[i], vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(short_name(cfg, name), fontsize=10); ax.axis('off')

plt.suptitle('Predicted Nutrient Concentrations — Field 14', fontsize=14)
plt.tight_layout()
fig.savefig(WORK_PATH / 'nutrient_maps_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Comparison: GBR baseline vs ResUNet ───────────────────────────────────
# baseline_cv_metrics.csv is created by the GBR baseline cell above
baseline_path = WORK_PATH / 'baseline_cv_metrics.csv'
if not baseline_path.exists():
    baseline_path = DATASET_PATH / 'baseline_cv_metrics.csv'  # uploaded from local run

if baseline_path.exists():
    df_base = pd.read_csv(baseline_path).set_index('element')
    df_cv   = pd.read_csv(WORK_PATH / 'cv_r2_summary.csv').set_index('element')

    print(f'{"Nutrient":<20} {"GBR":>8} {"ResUNet":>8} {"Δ":>6}')
    print('-'*46)
    for e in avail:
        sn  = short_name(cfg, e)
        r2b = df_base.loc[e, 'R2']    if e in df_base.index else np.nan
        r2u = df_cv.loc[e, 'R2_mean'] if e in df_cv.index   else np.nan
        d   = r2u - r2b if (np.isfinite(r2b) and np.isfinite(r2u)) else np.nan
        mark = '↑' if d > 0.02 else ('↓' if d < -0.02 else '≈')
        print(f'{sn:<20} {r2b:>+8.3f} {r2u:>+8.3f} {d:>+6.3f} {mark}')
else:
    print('baseline_cv_metrics.csv not found.')
    print('Either run the GBR baseline cell above, or upload baseline_cv_metrics.csv to the dataset.')
